# NHANES LSM Data Download

Downloads and prepares NHANES cycles that include liver stiffness measurement (LSM)
via FibroScan (vibration-controlled transient elastography, VCTE).

**Three LSM cycles:**

| Cycle | Label | Suffix/Prefix | Years collected | n (LUX exam) |
|-------|-------|--------------|-----------------|-------------|
| 2017–18 | `2017-2018` | `_J` suffix | Jan 2017 – Dec 2018 | ~6,400 |
| 2017–March 2020 Pre-Pandemic | `2017-2020` | `P_` prefix | Jan 2017 – Mar 2020 | ~9,000 |
| 2021–22 | `2021-2022` | `_L` suffix | Jan 2021 – Dec 2022 | ~7,200 |

**Important:** The 2017-2020 pre-pandemic cycle *supersedes and includes* the 2017-18 cycle.
Use either 2017-18 OR 2017-2020, not both together (they share participants).
For pseudo-cohort analyses, the pre-pandemic cycle is preferred because it includes
the January 2019 – March 2020 data that would otherwise be unavailable.

**Output:** Parquet files saved to `../data/derived/`:
- `2017_2018.parquet` — already produced by `nhanes_mortality_fibrosis/01_data_download.ipynb`
- `2017_2020_prepandemic.parquet` — produced by this notebook
- `2021_2022.parquet` — produced by this notebook

Run this notebook once; subsequent analysis notebooks load the parquets directly.

In [1]:
import os, hashlib
import requests
import numpy as np
import pandas as pd
import pyreadstat

DATA_DIR   = os.path.abspath(os.path.join('..', 'data'))
RAW_NHANES = os.path.join(DATA_DIR, 'raw', 'nhanes')
DERIVED    = os.path.join(DATA_DIR, 'derived')
os.makedirs(DERIVED, exist_ok=True)

NHANES_BASE = 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public'

RACE_MAP = {1: 'Mexican American', 2: 'Other Hispanic',
            3: 'Non-Hispanic White', 4: 'Non-Hispanic Black', 5: 'Other/Multi-Racial'}

In [2]:
def download(url, dest):
    """Download with caching; skip if file already exists."""
    if os.path.exists(dest):
        print(f'  cached: {os.path.basename(dest)}')
        return dest
    print(f'  GET {url}')
    r = requests.get(url, timeout=180)
    r.raise_for_status()
    with open(dest, 'wb') as f:
        f.write(r.content)
    md5 = hashlib.md5(r.content).hexdigest()
    print(f'    -> {os.path.basename(dest)} ({len(r.content):,} bytes, md5={md5})')
    return dest

def read_xpt(path):
    try:
        df, _ = pyreadstat.read_xport(path)
    except UnicodeDecodeError:
        df, _ = pyreadstat.read_xport(path, encoding='latin1')
    return df

## Cycle definitions

Each cycle spec carries enough info to build its download URLs:
- `year`: CDC server directory year
- `prefix`: file name prefix (empty string for suffix-style cycles)
- `suffix`: file name suffix (empty string for prefix-style cycles)
- `files`: list of component names; each resolves to `{prefix}{name}{suffix}.xpt`

In [3]:
CYCLES = [
    {
        'label':   '2017-2020',
        'dirname': '2017_2020_prepandemic',
        'year':    2017,
        'prefix':  'P_',    # e.g. P_DEMO.xpt
        'suffix':  '',
        'files':   ['DEMO', 'BIOPRO', 'CBC', 'BMX', 'BPXO', 'TRIGLY', 'GLU', 'SMQ', 'LUX'],
        'bpx_name': 'BPXO',  # blood pressure file name in this cycle
    },
    {
        'label':   '2021-2022',
        'dirname': '2021_2022',
        'year':    2021,
        'prefix':  '',
        'suffix':  '_L',    # e.g. DEMO_L.xpt
        'files':   ['DEMO', 'BIOPRO', 'CBC', 'BMX', 'BPXO', 'TRIGLY', 'GLU', 'SMQ', 'LUX'],
        'bpx_name': 'BPXO',
    },
]

def fname(cycle, component):
    return f"{cycle['prefix']}{component}{cycle['suffix']}.xpt"

def furl(cycle, component):
    return f"{NHANES_BASE}/{cycle['year']}/DataFiles/{fname(cycle, component)}"

## Download raw XPT files

In [4]:
raw = {}
for cyc in CYCLES:
    print(f"\n=== {cyc['label']} ===")
    cdir = os.path.join(RAW_NHANES, cyc['dirname'])
    os.makedirs(cdir, exist_ok=True)
    raw[cyc['label']] = {}
    for name in cyc['files']:
        path = download(furl(cyc, name), os.path.join(cdir, fname(cyc, name)))
        raw[cyc['label']][name] = read_xpt(path)
        print(f'    {name}: {len(raw[cyc["label"]][name]):,} rows')

print('\nAll downloads complete.')


=== 2017-2020 ===
  GET https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DEMO.xpt


    -> P_DEMO.xpt (3,614,720 bytes, md5=850382076eca1bef83b681d721c8362d)


    DEMO: 15,560 rows
  GET https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_BIOPRO.xpt


    -> P_BIOPRO.xpt (3,420,640 bytes, md5=c8751a52982cafc58235438866d3dff6)
    BIOPRO: 10,409 rows
  GET https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_CBC.xpt


    -> P_CBC.xpt (2,427,760 bytes, md5=498bb01dcc4f8ed6f66260aa85e97657)
    CBC: 13,772 rows
  GET https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_BMX.xpt


...

    -> P_TRIGLY.xpt (409,360 bytes, md5=e870c5f806d9bef8f8f7511ce2bde5c3)
    TRIGLY: 5,090 rows
  GET https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_GLU.xpt


    -> P_GLU.xpt (164,160 bytes, md5=b18d588f479a5d07053d480cb26a76a2)
    GLU: 5,090 rows
  GET https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_SMQ.xpt


    -> P_SMQ.xpt (1,428,560 bytes, md5=04ff912a72ff16dca1520c16718dfa64)
    SMQ: 11,137 rows
  GET https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_LUX.xpt


    -> P_LUX.xpt (1,105,920 bytes, md5=f13a60ec6b7dfb74ac70437f567ab9a0)
    LUX: 10,409 rows

=== 2021-2022 ===
  cached: DEMO_L.xpt
    DEMO: 11,933 rows
  cached: BIOPRO_L.xpt
    BIOPRO: 7,199 rows
  cached: CBC_L.xpt
    CBC: 8,727 rows
  cached: BMX_L.xpt
    BMX: 8,860 rows
  cached: BPXO_L.xpt
    BPXO: 7,801 rows
  cached: TRIGLY_L.xpt
    TRIGLY: 3,996 rows
  cached: GLU_L.xpt
    GLU: 3,996 rows
  cached: SMQ_L.xpt


    SMQ: 9,015 rows
  cached: LUX_L.xpt
    LUX: 7,199 rows

All downloads complete.


## Merge into analytic parquets

In [5]:
def get_col(df, *candidates):
    return next((c for c in candidates if c in df.columns), None)

def build_analytic(raw_d, cycle_label, bpx_name='BPX'):
    """Merge NHANES components for one LSM cycle into a wide analytic dataframe."""
    df = raw_d['DEMO'].copy()

    # AST and ALT
    bdf = raw_d['BIOPRO']
    ast_col = get_col(bdf, 'LBXSASSI', 'LBXSAS')
    alt_col = get_col(bdf, 'LBXSATSI', 'LBXSAT')
    df = df.merge(
        bdf[['SEQN', ast_col, alt_col]].rename(columns={ast_col: 'AST', alt_col: 'ALT'}),
        on='SEQN', how='left'
    )

    # Platelets
    cdf = raw_d['CBC']
    plt_col = get_col(cdf, 'LBXPLTSI', 'LBXPLT')
    df = df.merge(
        cdf[['SEQN', plt_col]].rename(columns={plt_col: 'PLATELETS'}),
        on='SEQN', how='left'
    )

    # BMI
    df = df.merge(raw_d['BMX'][['SEQN', 'BMXBMI']], on='SEQN', how='left')

    # Blood pressure — column names differ by cycle
    bpx = raw_d[bpx_name]
    sy_cols = [c for c in bpx.columns if 'SY' in c and c.startswith('BPX')]
    if sy_cols:
        bpx = bpx[['SEQN'] + sy_cols].copy()
        bpx['SBP_MEAN'] = bpx[sy_cols].mean(axis=1)
        df = df.merge(bpx[['SEQN', 'SBP_MEAN']], on='SEQN', how='left')
    else:
        df['SBP_MEAN'] = np.nan

    # Smoking
    smq = raw_d['SMQ'][['SEQN', 'SMQ020']].copy()
    smq['SMOKE_EVER'] = smq['SMQ020'].map({1: 1.0, 2: 0.0})
    df = df.merge(smq[['SEQN', 'SMOKE_EVER']], on='SEQN', how='left')

    # LSM (liver stiffness, kPa) and CAP (steatosis, dB/m)
    lux = raw_d['LUX'][['SEQN', 'LUXSMED', 'LUXCAPM', 'LUXSIQR']].rename(
        columns={'LUXSMED': 'LSM_KPA', 'LUXCAPM': 'CAP_DBM', 'LUXSIQR': 'LSM_IQR'}
    )
    df = df.merge(lux, on='SEQN', how='left')

    # Filter: exam-eligible adults ≥ 18
    df = df[df['RIDSTATR'] == 2].copy()
    df['AGE']    = df['RIDAGEYR']
    df = df[df['AGE'] >= 18].copy()
    df['FEMALE'] = (df['RIAGENDR'] == 2).astype(float)
    df['SEX']    = df['FEMALE'].map({0: 'Male', 1: 'Female'})
    df['RACE']   = df['RIDRETH1'].map(RACE_MAP)
    df['CYCLE']  = cycle_label

    # IQR/median ratio (reliability flag; >0.30 = unreliable per EASL guidance)
    df['LSM_IQR_RATIO'] = df['LSM_IQR'] / df['LSM_KPA']

    n     = len(df)
    lsm_n = df['LSM_KPA'].notna().sum()
    print(f'{cycle_label}: {n:,} eligible adults | LSM: {lsm_n:,} ({lsm_n/n*100:.0f}%)')
    return df

In [6]:
for cyc in CYCLES:
    label  = cyc['label']
    df_out = build_analytic(raw[label], label, bpx_name=cyc['bpx_name'])
    out    = os.path.join(DERIVED, f"{cyc['dirname']}.parquet")
    df_out.to_parquet(out, index=False)
    print(f'  -> {out} ({os.path.getsize(out)/1e6:.1f} MB)')

print('\nParquet files ready in data/derived/')

2017-2020: 8,965 eligible adults | LSM: 8,318 (93%)


  -> /home/abie/ai_assisted_us_health_data_analysis/data/derived/2017_2020_prepandemic.parquet (0.4 MB)


2021-2022: 6,337 eligible adults | LSM: 5,874 (93%)
  -> /home/abie/ai_assisted_us_health_data_analysis/data/derived/2021_2022.parquet (0.3 MB)

Parquet files ready in data/derived/


## Verify outputs

In [7]:
summary = []
for fname_stem in ['2017_2018', '2017_2020_prepandemic', '2021_2022']:
    path = os.path.join(DERIVED, f'{fname_stem}.parquet')
    if not os.path.exists(path):
        summary.append({'file': fname_stem, 'note': 'MISSING — run nhanes_mortality_fibrosis/01_data_download.ipynb first'})
        continue
    df = pd.read_parquet(path)
    lsm_n = df['LSM_KPA'].notna().sum() if 'LSM_KPA' in df.columns else 0
    summary.append({
        'file':     fname_stem,
        'n_adults': len(df),
        'lsm_n':    lsm_n,
        'lsm_pct':  f"{lsm_n/len(df)*100:.0f}%" if len(df) else 'n/a',
        'has_cap':  'CAP_DBM' in df.columns,
        'columns':  len(df.columns),
    })

display(pd.DataFrame(summary))
print('\nNote: 2017-2020 supersedes 2017-2018 — use one or the other, not both.')

,file,n_adults,lsm_n,lsm_pct,has_cap,columns
0,2017_2018,5809,5091,88%,True,64
1,2017_2020_prepandemic,8965,8318,93%,True,44
2,2021_2022,6337,5874,93%,True,42



Note: 2017-2020 supersedes 2017-2018 — use one or the other, not both.
